In [2]:
%load_ext autoreload
%autoreload 2

import torch

from utils import create_edge_loaders, create_hetero_graph
from gnn import model_factory
from train import train, evaluate, gnn_evaluate

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda

In [3]:
# Follow and activity data paths
FOLLOW_EDGELIST = 'data/higgs-social_network.edgelist'
ACTIVITY_EDGELIST = 'data/higgs-activity_time.txt'
target_interaction = "reply"

torch.manual_seed(42)

In [3]:
model_path = train(
    model_name="sage", 
    target_interaction=target_interaction, 
    activity_data_path=ACTIVITY_EDGELIST, 
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 202/202 [00:06<00:00, 28.96it/s, v_num=101, train_loss_step=0.669, train_acc_step=0.773, val_loss=0.927, val_auroc=0.805, val_acc=0.722, val_ap=0.608, val_recall=0.839, train_loss_epoch=0.854, train_acc_epoch=0.776]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 202/202 [00:06<00:00, 28.95it/s, v_num=101, train_loss_step=0.669, train_acc_step=0.773, val_loss=0.927, val_auroc=0.805, val_acc=0.722, val_ap=0.608, val_recall=0.839, train_loss_epoch=0.854, train_acc_epoch=0.776]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-sage-replyepoch=05-val_ap=0.6185.ckpt


In [4]:
# Evaluation
best_model_path = model_path
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
print(f"Evaluating best model from: {best_model_path}")

evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=0
)

Evaluating best model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-sage-replyepoch=05-val_ap=0.6186.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/44 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 44/44 [00:22<00:00,  1.97it/s]


--- Standard Metrics ---
Loss:   0.9635
AUROC:  0.8421
AP:     0.6204
Recall: 0.9162

--- Explanation Metrics (over 44 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.7500 (Higher is better)
Average Fidelity-: 0.6818 (Lower is better)


In [5]:
# Using GNNExplainer
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
gnn_evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 44/44 [00:00<00:00, 57.81it/s]



--- Standard Metrics ---
Loss:  0.9310
AUROC: 0.8481
AP:    0.6310

--- Setting up Explainer (Option B) ---


Explaining:   0%|          | 0/5 [00:00<?, ?it/s]


--- Top Explanations for Target Edge 0 ---
  Top 5 important 'FL' edges:
    User 21 -> User 19 (Weight: 0.1342)
    User 19 -> User 15 (Weight: 0.1341)
    User 127 -> User 9 (Weight: 0.1333)
    User 6 -> User 16 (Weight: 0.1331)
    User 180 -> User 13 (Weight: 0.1329)
  Top 5 important 'RT' edges:
    User 288 -> User 23 (Weight: 0.9290)
    User 274 -> User 5 (Weight: 0.0831)
    User 259 -> User 2 (Weight: 0.0831)
    User 280 -> User 5 (Weight: 0.0830)
    User 281 -> User 5 (Weight: 0.0830)
  Top 5 important 'MT' edges:
    User 23 -> User 1 (Weight: 0.1214)
    User 296 -> User 2 (Weight: 0.0821)
    User 69 -> User 4 (Weight: 0.0820)
    User 7 -> User 6 (Weight: 0.0820)
    User 84 -> User 21 (Weight: 0.0820)
  Top 5 important 'RE' edges:
    User 325 -> User 2 (Weight: 0.9332)
    User 320 -> User 2 (Weight: 0.9331)
    User 267 -> User 2 (Weight: 0.9331)
    User 23 -> User 1 (Weight: 0.0839)
    User 323 -> User 2 (Weight: 0.0772)


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 51 -> User 1 (Weight: 0.1470)
    User 645 -> User 45 (Weight: 0.1465)
    User 861 -> User 59 (Weight: 0.1463)
    User 678 -> User 47 (Weight: 0.1461)
    User 723 -> User 49 (Weight: 0.1456)
  Top 5 important 'RT' edges:
    User 43 -> User 1 (Weight: 0.1609)
    User 914 -> User 31 (Weight: 0.1055)
    User 668 -> User 46 (Weight: 0.1051)
    User 1013 -> User 52 (Weight: 0.1049)
    User 905 -> User 21 (Weight: 0.1049)
  Top 5 important 'MT' edges:
    User 62 -> User 0 (Weight: 0.1044)
    User 63 -> User 0 (Weight: 0.1038)
    User 1045 -> User 10 (Weight: 0.0981)
    User 1088 -> User 50 (Weight: 0.0976)
    User 1048 -> User 24 (Weight: 0.0975)
  Top 5 important 'RE' edges:
    User 1056 -> User 33 (Weight: 0.0816)
    User 1072 -> User 45 (Weight: 0.0816)
    User 1053 -> User 31 (Weight: 0.0816)
    User 1051 -> User 25 (Weight: 0.0815)
    User 1067 -> User 45 (Weight: 0.0815)


Explaining:  80%|████████  | 4/5 [00:10<00:02,  2.52s/it]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 238 -> User 8 (Weight: 0.1500)
    User 551 -> User 30 (Weight: 0.1471)
    User 654 -> User 38 (Weight: 0.1466)
    User 321 -> User 14 (Weight: 0.1464)
    User 775 -> User 116 (Weight: 0.1463)
  Top 5 important 'RT' edges:
    User 1076 -> User 59 (Weight: 0.0920)
    User 71 -> User 1 (Weight: 0.0918)
    User 90 -> User 1 (Weight: 0.0917)
    User 82 -> User 1 (Weight: 0.0916)
    User 88 -> User 1 (Weight: 0.0916)
  Top 5 important 'MT' edges:
    User 100 -> User 1 (Weight: 0.0877)
    User 97 -> User 1 (Weight: 0.0877)
    User 119 -> User 1 (Weight: 0.0876)
    User 1094 -> User 21 (Weight: 0.0874)
    User 1100 -> User 86 (Weight: 0.0874)
  Top 5 important 'RE' edges:
    User 1100 -> User 86 (Weight: 0.0788)
    User 97 -> User 1 (Weight: 0.0786)
    User 122 -> User 1 (Weight: 0.0786)
    User 672 -> User 40 (Weight: 0.0786)
    User 119 -> User 1 (Weight: 0.0785)


Explaining: 100%|██████████| 5/5 [00:12<00:00,  2.58s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 756 -> User 48 (Weight: 0.1505)
    User 589 -> User 34 (Weight: 0.1476)
    User 1060 -> User 105 (Weight: 0.1475)
    User 871 -> User 58 (Weight: 0.1474)
    User 39 -> User 1 (Weight: 0.1473)
  Top 5 important 'RT' edges:
    User 1137 -> User 21 (Weight: 0.1084)
    User 1144 -> User 31 (Weight: 0.1076)
    User 1110 -> User 11 (Weight: 0.1066)
    User 1123 -> User 21 (Weight: 0.1066)
    User 1172 -> User 40 (Weight: 0.1064)
  Top 5 important 'MT' edges:
    User 1214 -> User 11 (Weight: 0.1026)
    User 1228 -> User 11 (Weight: 0.1021)
    User 1238 -> User 21 (Weight: 0.1015)
    User 1211 -> User 9 (Weight: 0.1012)
    User 93 -> User 0 (Weight: 0.1010)
  Top 5 important 'RE' edges:
    User 1276 -> User 76 (Weight: 0.0839)
    User 1213 -> User 11 (Weight: 0.0839)
    User 1273 -> User 53 (Weight: 0.0838)
    User 1221 -> User 11 (Weight: 0.0838)
    User 1292 -> User 32 (Weight: 0.0838)

--- 

In [3]:
# Comparison with vanilla GNN
vanilla_model_path = train(
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 202/202 [00:03<00:00, 58.45it/s, v_num=100, train_loss_step=0.621, train_acc_step=0.841, val_loss=0.665, val_auroc=0.705, val_acc=0.794, val_ap=0.579, val_recall=0.434, train_loss_epoch=0.645, train_acc_epoch=0.836]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 202/202 [00:03<00:00, 58.40it/s, v_num=100, train_loss_step=0.621, train_acc_step=0.841, val_loss=0.665, val_auroc=0.705, val_acc=0.794, val_ap=0.579, val_recall=0.434, train_loss_epoch=0.645, train_acc_epoch=0.836]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-simple-replyepoch=07-val_ap=0.6077.ckpt


In [7]:
# vanilla_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-simple-replyepoch=07-val_ap=0.6077.ckpt"
print(f"Evaluating vanilla GNN model from: {vanilla_model_path}")
evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=0
)
gnn_evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating vanilla GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-simple-replyepoch=07-val_ap=0.6077.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/44 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 44/44 [00:20<00:00,  2.12it/s]



--- Standard Metrics ---
Loss:   0.6595
AUROC:  0.7302
AP:     0.6200
Recall: 0.4881

--- Explanation Metrics (over 44 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.5455 (Higher is better)
Average Fidelity-: 0.5000 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 44/44 [00:04<00:00, 10.20it/s]



--- Standard Metrics ---
Loss:  0.6578
AUROC: 0.7292
AP:    0.6165

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 27 -> User 0 (Weight: 0.7064)
    User 46 -> User 1 (Weight: 0.7020)
    User 53 -> User 1 (Weight: 0.7017)
    User 45 -> User 1 (Weight: 0.7006)
    User 47 -> User 1 (Weight: 0.6874)
  Top 5 important 'RT' edges:
    User 99 -> User 106 (Weight: 0.6920)
    User 1227 -> User 92 (Weight: 0.6878)
    User 1249 -> User 106 (Weight: 0.6735)
    User 269 -> User 106 (Weight: 0.6667)
    User 1246 -> User 102 (Weight: 0.5745)
  Top 5 important 'MT' edges:
    User 1295 -> User 106 (Weight: 0.6416)
    User 106 -> User 1 (Weight: 0.6137)
    User 102 -> User 1 (Weight: 0.6022)
    User 1298 -> User 106 (Weight: 0.5969)
    User 92 -> User 1 (Weight: 0.5968)
  Top 5 important 'RE' edges:
    User 91 -> User 1 (Weight: 0.5813)
    User 100 -> User 1 (Weight: 0.5695)
    User 95 -> User 1 (Weight: 0.5594)
    User 118 -> User 1 (Weight: 0.5545)
    User 113 -> User 1 (Weight: 0.5535)


Explaining:  80%|████████  | 4/5 [01:18<00:29, 29.38s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 984 -> User 110 (Weight: 0.7230)
    User 28 -> User 0 (Weight: 0.6893)
    User 11 -> User 0 (Weight: 0.6831)
    User 5 -> User 0 (Weight: 0.6773)
    User 27 -> User 0 (Weight: 0.6604)
  Top 5 important 'RT' edges:
    User 1077 -> User 50 (Weight: 0.9130)
    User 1062 -> User 32 (Weight: 0.9005)
    User 1079 -> User 50 (Weight: 0.8683)
    User 1076 -> User 50 (Weight: 0.8626)
    User 1083 -> User 51 (Weight: 0.8476)
  Top 5 important 'MT' edges:
    User 1105 -> User 27 (Weight: 0.9045)
    User 1132 -> User 53 (Weight: 0.8976)
    User 1128 -> User 50 (Weight: 0.8739)
    User 1135 -> User 60 (Weight: 0.8338)
    User 99 -> User 0 (Weight: 0.7624)
  Top 5 important 'RE' edges:
    User 1106 -> User 32 (Weight: 0.9336)
    User 1147 -> User 32 (Weight: 0.9326)
    User 116 -> User 0 (Weight: 0.5457)
    User 112 -> User 0 (Weight: 0.5234)
    User 115 -> User 0 (Weight: 0.5133)


Explaining: 100%|██████████| 5/5 [01:26<00:00, 17.33s/it]


--- Explanation Metrics (over 5 edges) ---
Average Fidelity+: 0.0000 (Higher is better)
Average Fidelity-: 0.0000 (Lower is better)


In [4]:
# Ablation study: stripped GNN
stripped_model_path = train(
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 202/202 [00:02<00:00, 76.48it/s, v_num=95, train_loss_step=1.020, train_acc_step=0.819, val_loss=0.946, val_auroc=0.631, val_acc=0.728, val_ap=0.432, val_recall=0.349, train_loss_epoch=0.862, train_acc_epoch=0.834]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 202/202 [00:02<00:00, 76.41it/s, v_num=95, train_loss_step=1.020, train_acc_step=0.819, val_loss=0.946, val_auroc=0.631, val_acc=0.728, val_ap=0.432, val_recall=0.349, train_loss_epoch=0.862, train_acc_epoch=0.834]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-stripped-replyepoch=03-val_ap=0.5584.ckpt


In [5]:
stripped_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-stripped-replyepoch=03-val_ap=0.5584.ckpt"
print(f"Evaluating stripped GNN model from: {stripped_model_path}")
evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating stripped GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_reply/best-gnn-stripped-replyepoch=03-val_ap=0.5584.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/44 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   2%|▏         | 1/44 [00:01<01:22,  1.92s/it]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   7%|▋         | 3/44 [00:02<00:26,  1.55it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 44/44 [00:09<00:00,  4.57it/s]



--- Standard Metrics ---
Loss:   1.1158
AUROC:  0.7859
AP:     0.5672
Recall: 0.8120

--- Explanation Metrics (over 44 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.7500 (Higher is better)
Average Fidelity-: 0.6818 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 44/44 [00:02<00:00, 15.56it/s]



--- Standard Metrics ---
Loss:  1.1262
AUROC: 0.7843
AP:    0.5638

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 51 -> User 1 (Weight: 0.0881)
    User 32 -> User 1 (Weight: 0.0879)
    User 40 -> User 1 (Weight: 0.0879)
    User 49 -> User 1 (Weight: 0.0879)
    User 33 -> User 1 (Weight: 0.0878)
  Top 5 important 'RT' edges:
    User 46 -> User 1 (Weight: 0.9284)
    User 887 -> User 4 (Weight: 0.0000)
    User 885 -> User 4 (Weight: 0.0000)
    User 888 -> User 4 (Weight: 0.0000)
    User 886 -> User 4 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 63 -> User 0 (Weight: 0.9287)
    User 62 -> User 0 (Weight: 0.0725)
    User 956 -> User 4 (Weight: 0.0000)
    User 886 -> User 4 (Weight: 0.0000)
    User 955 -> User 2 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.0721)
    User 969 -> User 23 (Weight: 0.0000)
    User 955 -> User 2 (Weight: 0.0000)
    User 973 -> User 42 (Weight: 0.0000)
    User 965 -> User 18 (Weight: 0.0000)


Explaining:  80%|████████  | 4/5 [00:03<00:00,  1.17it/s]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 45 -> User 1 (Weight: 0.0879)
    User 23 -> User 0 (Weight: 0.0878)
    User 56 -> User 1 (Weight: 0.0876)
    User 59 -> User 1 (Weight: 0.0876)
    User 49 -> User 1 (Weight: 0.0876)
  Top 5 important 'RT' edges:
    User 71 -> User 1 (Weight: 0.0800)
    User 90 -> User 1 (Weight: 0.0800)
    User 82 -> User 1 (Weight: 0.0799)
    User 86 -> User 1 (Weight: 0.0799)
    User 88 -> User 1 (Weight: 0.0799)
  Top 5 important 'MT' edges:
    User 100 -> User 1 (Weight: 0.0800)
    User 97 -> User 1 (Weight: 0.0800)
    User 118 -> User 1 (Weight: 0.0800)
    User 116 -> User 1 (Weight: 0.0799)
    User 96 -> User 1 (Weight: 0.0799)
  Top 5 important 'RE' edges:
    User 92 -> User 1 (Weight: 0.0749)
    User 118 -> User 1 (Weight: 0.0749)
    User 115 -> User 1 (Weight: 0.0748)
    User 99 -> User 1 (Weight: 0.0748)
    User 101 -> User 1 (Weight: 0.0748)


Explaining: 100%|██████████| 5/5 [00:04<00:00,  1.15it/s]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 39 -> User 1 (Weight: 0.0867)
    User 51 -> User 1 (Weight: 0.0866)
    User 15 -> User 0 (Weight: 0.0864)
    User 4 -> User 0 (Weight: 0.0864)
    User 50 -> User 1 (Weight: 0.0863)
  Top 5 important 'RT' edges:
    User 80 -> User 0 (Weight: 0.0799)
    User 77 -> User 0 (Weight: 0.0799)
    User 61 -> User 0 (Weight: 0.0799)
    User 73 -> User 0 (Weight: 0.0798)
    User 76 -> User 0 (Weight: 0.0798)
  Top 5 important 'MT' edges:
    User 93 -> User 0 (Weight: 0.0800)
    User 97 -> User 0 (Weight: 0.0799)
    User 99 -> User 0 (Weight: 0.0799)
    User 101 -> User 0 (Weight: 0.0799)
    User 102 -> User 0 (Weight: 0.0799)
  Top 5 important 'RE' edges:
    User 116 -> User 0 (Weight: 0.9288)
    User 114 -> User 0 (Weight: 0.0726)
    User 115 -> User 0 (Weight: 0.0726)
    User 1207 -> User 14 (Weight: 0.0000)
    User 1207 -> User 14 (Weight: 0.0000)

--- Explanation Metrics (over 5 edges) ---
Av